### LightGBM

In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, precision_score, recall_score, f1_score,
                             classification_report, precision_recall_curve)
from sklearn.calibration import CalibratedClassifierCV

ind_hosp = pd.read_parquet('ind_hosp.parquet', engine='fastparquet')
ind_hosp['dischtime'] = pd.to_datetime(ind_hosp['dischtime'])

X = ind_hosp.drop(columns=['subject_id', 'hadm_id', 'readmission_30day', 'dischtime'])
y = ind_hosp['readmission_30day']

bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

patient_last_discharge = ind_hosp.groupby('subject_id')['dischtime'].max().reset_index()
patient_last_discharge = patient_last_discharge.sort_values('dischtime').reset_index(drop=True)

n_patients = len(patient_last_discharge)
train_patients_end = int(0.7 * n_patients)
val_patients_end = int(0.85 * n_patients)

train_patients = set(patient_last_discharge.iloc[:train_patients_end]['subject_id'])
val_patients = set(patient_last_discharge.iloc[train_patients_end:val_patients_end]['subject_id'])
test_patients = set(patient_last_discharge.iloc[val_patients_end:]['subject_id'])

train_mask = ind_hosp['subject_id'].isin(train_patients)
val_mask = ind_hosp['subject_id'].isin(val_patients)
test_mask = ind_hosp['subject_id'].isin(test_patients)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Patient-wise chronological order:")
print(f"Train: {X_train.shape[0]} rows, patients: {len(train_patients)}")
print(f"Validation: {X_val.shape[0]} rows, patients: {len(val_patients)}")
print(f"Test: {X_test.shape[0]} rows, patients: {len(test_patients)}")
print(f"Train readmission rate: {y_train.mean():.2%}")
print(f"Validation readmission rate: {y_val.mean():.2%}")
print(f"Test readmission rate: {y_test.mean():.2%}")

params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'scale_pos_weight': (1 - y_train.mean()) / y_train.mean(),
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

print(f"\nScale pos weight: {params['scale_pos_weight']:.2f}")

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

calibrated = CalibratedClassifierCV(model, method='sigmoid', cv='prefit')
calibrated.fit(X_val, y_val)

y_val_proba_calibrated = calibrated.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba_calibrated)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]
print(f"\nOptimal threshold after calibration: {best_threshold:.4f}")

y_pred_proba = calibrated.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= best_threshold).astype(int)

auc_roc = roc_auc_score(y_test, y_pred_proba)
auc_prc = average_precision_score(y_test, y_pred_proba)
brier = brier_score_loss(y_test, y_pred_proba)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"LightGBM results")
print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PRC: {auc_prc:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"\nAt threshold {best_threshold}:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Negative Predictive Value: {npv:.4f}")
print(f"F1-Score: {f1:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No readmission', 'Readmission']))

importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 20 features (LightGBM):")
print(importances.head(20).to_string(index=False))

Patient-wise chronological order:
Train: 275237 rows, patients: 127060
Validation: 62199 rows, patients: 27227
Test: 77592 rows, patients: 27228
Train readmission rate: 19.25%
Validation readmission rate: 19.33%
Test readmission rate: 20.26%

Scale pos weight: 4.19
Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.71873
[100]	valid_0's auc: 0.723183
[150]	valid_0's auc: 0.724141
[200]	valid_0's auc: 0.724319
[250]	valid_0's auc: 0.724217
Early stopping, best iteration is:
[210]	valid_0's auc: 0.724396


/Users/arinapetuhova/GitHub/tusted_llm_risk_profiling/.venv/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(



Optimal threshold after calibration: 0.2128
LightGBM results
AUC-ROC: 0.7182
AUC-PRC: 0.4033
Brier Score: 0.1446

At threshold 0.21275465432775426:
Accuracy: 0.6755
Precision: 0.3389
Recall: 0.6326
Specificity: 0.6864
Negative Predictive Value: 0.8803
F1-Score: 0.4413

Classification Report:
                precision    recall  f1-score   support

No readmission       0.88      0.69      0.77     61871
   Readmission       0.34      0.63      0.44     15721

      accuracy                           0.68     77592
     macro avg       0.61      0.66      0.61     77592
  weighted avg       0.77      0.68      0.70     77592


Top 20 features (LightGBM):
              feature  importance
                  age         448
      num_medications         415
prior_admissions_12mo         342
       lab_51301_last         250
    comorbidity_score         213
       num_procedures         210
       lab_50983_last         202
       lab_51221_last         201
       lab_50931_last         20

### XGBoost

In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, precision_score, recall_score, f1_score,
                             classification_report, precision_recall_curve)
from sklearn.calibration import CalibratedClassifierCV

ind_hosp = pd.read_parquet('ind_hosp.parquet', engine='fastparquet')
ind_hosp['dischtime'] = pd.to_datetime(ind_hosp['dischtime'])

X = ind_hosp.drop(columns=['subject_id', 'hadm_id', 'readmission_30day', 'dischtime'])
y = ind_hosp['readmission_30day']

bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

patient_last_discharge = ind_hosp.groupby('subject_id')['dischtime'].max().reset_index()
patient_last_discharge = patient_last_discharge.sort_values('dischtime').reset_index(drop=True)

n_patients = len(patient_last_discharge)
train_patients_end = int(0.7 * n_patients)
val_patients_end = int(0.85 * n_patients)

train_patients = set(patient_last_discharge.iloc[:train_patients_end]['subject_id'])
val_patients = set(patient_last_discharge.iloc[train_patients_end:val_patients_end]['subject_id'])
test_patients = set(patient_last_discharge.iloc[val_patients_end:]['subject_id'])

train_mask = ind_hosp['subject_id'].isin(train_patients)
val_mask = ind_hosp['subject_id'].isin(val_patients)
test_mask = ind_hosp['subject_id'].isin(test_patients)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Patient-wise chronological order:")
print(f"Train: {X_train.shape[0]} rows, patients: {len(train_patients)}")
print(f"Validation: {X_val.shape[0]} rows, patients: {len(val_patients)}")
print(f"Test: {X_test.shape[0]} rows, patients: {len(test_patients)}")
print(f"Train readmission rate: {y_train.mean():.2%}")
print(f"Validation readmission rate: {y_val.mean():.2%}")
print(f"Test readmission rate: {y_test.mean():.2%}")

scale_pos_weight = (1 - y_train.mean()) / y_train.mean()

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'scale_pos_weight': scale_pos_weight,
    'min_child_weight': 3,
    'gamma': 0.2,
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': -1,
    'early_stopping_rounds': 50
}

print(f"\nScale pos weight: {params['scale_pos_weight']:.2f}")

model = xgb.XGBClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)
calibrated = CalibratedClassifierCV(model, method='sigmoid', cv='prefit')
calibrated.fit(X_val, y_val)

y_val_proba_calibrated = calibrated.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba_calibrated)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]
print(f"\nOptimal threshold after calibration: {best_threshold:.4f}")

y_pred_proba = calibrated.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= best_threshold).astype(int)

auc_roc = roc_auc_score(y_test, y_pred_proba)
auc_prc = average_precision_score(y_test, y_pred_proba)
brier = brier_score_loss(y_test, y_pred_proba)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"XGBoost Results")
print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PRC: {auc_prc:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"\nAt threshold {best_threshold}:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Negative Predictive Value: {npv:.4f}")
print(f"F1-Score: {f1:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No readmission', 'Readmission']))

importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 20 features:")
print(importances.head(20).to_string(index=False))

Patient-wise chronological order:
Train: 275237 rows, patients: 127060
Validation: 62199 rows, patients: 27227
Test: 77592 rows, patients: 27228
Train readmission rate: 19.25%
Validation readmission rate: 19.33%
Test readmission rate: 20.26%

Scale pos weight: 4.19
[0]	validation_0-auc:0.68597
[50]	validation_0-auc:0.71363
[100]	validation_0-auc:0.71919
[150]	validation_0-auc:0.72197
[200]	validation_0-auc:0.72318
[250]	validation_0-auc:0.72376
[299]	validation_0-auc:0.72436

Optimal threshold after calibration: 0.2258


/Users/arinapetuhova/GitHub/tusted_llm_risk_profiling/.venv/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


XGBoost Results
AUC-ROC: 0.7181
AUC-PRC: 0.4036
Brier Score: 0.1445

At threshold 0.22580271474230001:
Accuracy: 0.6939
Precision: 0.3494
Recall: 0.5928
Specificity: 0.7196
Negative Predictive Value: 0.8743
F1-Score: 0.4397

Classification Report:
                precision    recall  f1-score   support

No readmission       0.87      0.72      0.79     61871
   Readmission       0.35      0.59      0.44     15721

      accuracy                           0.69     77592
     macro avg       0.61      0.66      0.61     77592
  weighted avg       0.77      0.69      0.72     77592


Top 20 features:
                                   feature  importance
                     prior_admissions_12mo    0.090506
                             lab_51277_max    0.039479
                   discharge_location_HOME    0.033784
                           lab_50893_count    0.029962
                         comorbidity_score    0.028266
                              race_UNKNOWN    0.025263
         d